# 2025 Global Box Office Analysis with NumPy and pandas

**Data source:** [Box Office Mojo — 2025 Worldwide](https://www.boxofficemojo.com/year/world/2025/)

**Goal:** Practise the core NumPy and pandas skills from Session 7 by analysing real-world box-office data stored in a JSON file.

We will load the data into a **DataFrame**, clean the percentage columns, and then explore, filter, sort, and summarise the dataset using the techniques covered in class.

## Outline

0. **Setup** — import NumPy and pandas
1. **Loading and Inspecting the Data** — read JSON into a DataFrame; `.shape`, `.dtypes`, `.head()`
2. **Data Cleaning** — convert `domestic_share` and `foreign_share` from strings to numbers
3. **Selecting and Indexing** — `.loc`, `.iloc`, column selection
4. **Filtering with Boolean Conditions** — single and multiple conditions
5. **Sorting and Ranking** — `.sort_values()`, `.rank()`
6. **Column Arithmetic** — calculate domestic and foreign revenue
7. **Mapping and Apply** — `.map()`, `.apply()`, `np.where()` for categorisation
8. **Summary Statistics** — `.describe()`, `.value_counts()`, `.unique()`, `.isin()`
9. **Correlation** — `.corr()` between numeric columns
10. **NumPy Integration** — `.to_numpy()`, vectorised operations

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import json

---
## 1. Loading and Inspecting the Data

The JSON file contains a **list of dictionaries** — one dictionary per film. Passing that list straight to `pd.DataFrame()` produces a table where each dictionary key becomes a column name.

> **Recall from Session 7:** A DataFrame can be created from a *dict of lists* or a *list of dicts*. Our JSON data is already in the second format, so pandas handles the conversion automatically.

In [ ]:
df = pd.read_json('box_office_2025.json')
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")
print()
df.head(10)

In [ ]:
print("Data types BEFORE cleaning:")
print(df.dtypes)
print()
print("Sample domestic_share values:", df["domestic_share"].unique()[:8])

---
## 2. Data Cleaning — Converting Percentage Strings to Numbers

The `domestic_share` and `foreign_share` columns are stored as **strings** like `"42.3%"` or `"-"`. Before we can do any numerical analysis we need to convert them to **float** values.

**Strategy:**
1. Use **`.map()`** with a custom function to strip the `%` sign and convert to `float`.
2. Values that cannot be converted (`"-"`, `"<0.1%"`) become `NaN` — pandas' standard marker for missing data.

> **Session 7 connection:** This is a real-world example of `.map(func)` — applying a function to every value in a Series.

In [ ]:
def parse_percentage(value):
    """Convert a percentage string like '42.3%' to a float (42.3).
    Returns NaN for non-numeric entries such as '-' or '<0.1%'."""
    if value in ("-", "<0.1%"):
        return np.nan
    return float(value.replace("%", ""))

df["domestic_pct"] = df["domestic_share"].map(parse_percentage)
df["foreign_pct"] = df["foreign_share"].map(parse_percentage)

print("Data types AFTER cleaning:")
print(df[["domestic_share", "domestic_pct", "foreign_share", "foreign_pct"]].dtypes)
print()
df[["title", "domestic_share", "domestic_pct", "foreign_share", "foreign_pct"]].head(10)

In [ ]:
valid_count = df["domestic_pct"].notna().sum()
missing_count = df["domestic_pct"].isna().sum()

print(f"Films with valid domestic percentage: {valid_count}")
print(f"Films with missing data (NaN):        {missing_count}")
print(f"Total:                                 {len(df)}")

---
## 3. Selecting and Indexing

pandas provides three main ways to select data:

| Method | Purpose |
|--------|---------|
| `df["col"]` | Select a single column (returns a Series) |
| `df.loc[rows, cols]` | Select by **label** |
| `df.iloc[rows, cols]` | Select by **integer position** |

In [ ]:
print("Select a single column (Series):")
print(df["title"].head())
print()

print("Select by label — .loc (first 3 rows, specific columns):")
print(df.loc[0:2, ["title", "worldwide_gross", "domestic_pct"]])
print()

print("Select by position — .iloc (rows 0–2, columns 0–3):")
print(df.iloc[0:3, 0:3])

---
## 4. Filtering with Boolean Conditions

Just like NumPy Boolean filtering, we can pass a condition inside `df[...]` to keep only the rows that satisfy it.

> **Reminder:** For multiple conditions, use `&` (AND), `|` (OR), and `~` (NOT) — **not** the Python keywords `and`, `or`, `not`. Wrap each condition in parentheses.

In [ ]:
billion_club = df[df["worldwide_gross"] >= 1_000_000_000]
print(f"Films that crossed $1 billion: {len(billion_club)}\n")
print(billion_club[["rank", "title", "worldwide_gross"]])

In [ ]:
foreign_dominant = df[(df["foreign_pct"] >= 70) & (df["worldwide_gross"] >= 500_000_000)]
print("Foreign-dominant blockbusters (foreign ≥ 70% AND gross ≥ $500M):\n")
print(foreign_dominant[["title", "worldwide_gross", "foreign_pct"]].to_string(index=False))

In [ ]:
domestic_hits = df[~(df["domestic_pct"] < 50)]
print(f"Films where domestic share is NOT below 50%: {len(domestic_hits)}\n")
print(domestic_hits[["title", "domestic_pct", "worldwide_gross"]].head(10))

---
## 5. Sorting and Ranking

| Method | What it does |
|--------|-------------|
| `.sort_values(col)` | Reorder rows by a column's **values** |
| `.sort_index()` | Reorder rows by their **index labels** |
| `.rank()` | Assign a rank (1st, 2nd, …) without reordering |

In [ ]:
print("=== Top 10 Films Sorted by Domestic Share (highest first) ===\n")
top_domestic = df.sort_values("domestic_pct", ascending=False).head(10)
print(top_domestic[["title", "worldwide_gross", "domestic_pct"]].to_string(index=False))

In [ ]:
print("=== Top 10 Films Sorted by Foreign Share (highest first) ===\n")
top_foreign = df.sort_values("foreign_pct", ascending=False).head(10)
print(top_foreign[["title", "worldwide_gross", "foreign_pct"]].to_string(index=False))

In [ ]:
ranked = df[["title", "worldwide_gross"]].copy()
ranked["gross_rank"] = ranked["worldwide_gross"].rank(ascending=False).astype(int)
ranked["domestic_rank"] = df["domestic_pct"].rank(ascending=False, na_option="bottom").astype(int)

print("Ranking comparison — gross rank vs domestic-share rank:\n")
print(ranked.sort_values("gross_rank").head(15).to_string(index=False))

---
## 6. Column Arithmetic — Calculating Revenue Breakdown

Because `domestic_pct` and `foreign_pct` are now numeric, we can use **vectorised column arithmetic** (just like NumPy) to compute the dollar amounts earned in each market.

> **Session 7 connection:** Arithmetic between DataFrame columns is element-wise, the same as NumPy array arithmetic — no loops required.

In [ ]:
df["domestic_revenue"] = (df["worldwide_gross"] * df["domestic_pct"] / 100).round(0)
df["foreign_revenue"] = (df["worldwide_gross"] * df["foreign_pct"] / 100).round(0)

print("Revenue breakdown for the top 10 films:\n")
cols = ["title", "worldwide_gross", "domestic_revenue", "foreign_revenue"]
df[cols].head(10)

In [ ]:
df["gross_in_billions"] = df["worldwide_gross"] / 1_000_000_000

print("Quick check — gross in billions for the top 5:")
print(df[["title", "worldwide_gross", "gross_in_billions"]].head())

---
## 7. Mapping, Apply, and `np.where()` — Categorising Films

| Tool | Use case |
|------|----------|
| `.map(dict)` | Replace values via a lookup dictionary |
| `.map(func)` | Transform every value with a function |
| `.apply(func)` | Run a function across rows or columns |
| `np.where(cond, A, B)` | Conditional column — like Excel's `IF` |

In [ ]:
def revenue_tier(gross):
    """Classify a film into a revenue tier."""
    if gross >= 1_000_000_000:
        return "$1B+"
    elif gross >= 500_000_000:
        return "$500M–$1B"
    elif gross >= 100_000_000:
        return "$100M–$500M"
    elif gross >= 50_000_000:
        return "$50M–$100M"
    else:
        return "Under $50M"

df["tier"] = df["worldwide_gross"].map(revenue_tier)

print(".map(func) — revenue tier for the first 12 films:\n")
print(df[["title", "worldwide_gross", "tier"]].head(12).to_string(index=False))

In [ ]:
df["market_type"] = np.where(
    df["domestic_pct"] >= 50, "Domestic-led", "Foreign-led"
)

print("np.where() — market type based on domestic share:\n")
print(df[["title", "domestic_pct", "market_type"]].head(15).to_string(index=False))

In [ ]:
print(".apply(len) — title lengths:\n")
df["title_length"] = df["title"].apply(len)
print(df[["title", "title_length"]].sort_values("title_length", ascending=False).head(10).to_string(index=False))
print()

print("Range of worldwide_gross (max − min) via .apply():")
print(df[["worldwide_gross", "domestic_revenue", "foreign_revenue"]].apply(
    lambda col: col.max() - col.min()
))

---
## 8. Summary Statistics

| Method | What it does |
|--------|-------------|
| `.describe()` | Count, mean, std, min, quartiles, max for numeric columns |
| `.value_counts()` | Frequency of each unique value (like `collections.Counter`) |
| `.unique()` | Distinct values (like converting a list to a set) |
| `.isin(list)` | Boolean mask — `True` for values found in the given list |

In [ ]:
print(".describe() — summary statistics for key numeric columns:\n")
df[["worldwide_gross", "domestic_pct", "foreign_pct"]].describe()

In [ ]:
print(".value_counts() — number of films in each revenue tier:\n")
print(df["tier"].value_counts())
print()

print(".value_counts() — market type distribution:\n")
print(df["market_type"].value_counts())

In [ ]:
print(".unique() — distinct revenue tiers:\n")
print(df["tier"].unique())
print()

target_tiers = ["$1B+", "$500M–$1B"]
big_films = df[df["tier"].isin(target_tiers)]
print(f".isin() — films in the {target_tiers} tiers: {len(big_films)}\n")
print(big_films[["title", "worldwide_gross", "tier"]].to_string(index=False))

---
## 9. Correlation

Correlation measures the strength of a **linear relationship** between two numeric columns. Values range from −1 (perfect negative) to +1 (perfect positive).

> **Session 7 connection:** `.corr()` works on a single pair of Series or on an entire DataFrame to produce a correlation matrix.

In [ ]:
corr_value = df["domestic_pct"].corr(df["worldwide_gross"])
print(f"Correlation between domestic share and worldwide gross: {corr_value:.4f}")
print("(A negative value suggests that the biggest earners tend to skew towards foreign markets.)\n")

print("Full correlation matrix:\n")
df[["worldwide_gross", "domestic_pct", "foreign_pct", "domestic_revenue", "foreign_revenue"]].corr()

---
## 10. NumPy Integration — Vectorised Analysis

pandas is built on top of NumPy. We can extract a NumPy array from any DataFrame column using `.to_numpy()` and then apply fast, vectorised operations.

> **Session 7 connection:** This demonstrates the pandas → NumPy → pandas workflow — use pandas for labelled data, drop to NumPy for raw numerical computation, and bring results back into a DataFrame.

In [ ]:
gross_array = df["worldwide_gross"].to_numpy()

print(f"Type:  {type(gross_array)}")
print(f"Dtype: {gross_array.dtype}")
print(f"Shape: {gross_array.shape}")
print()

print("NumPy summary statistics (no loops needed):")
print(f"  Sum:    ${gross_array.sum():,.0f}")
print(f"  Mean:   ${gross_array.mean():,.0f}")
print(f"  Median: ${np.median(gross_array):,.0f}")
print(f"  Std:    ${gross_array.std():,.0f}")
print(f"  Min:    ${gross_array.min():,.0f}")
print(f"  Max:    ${gross_array.max():,.0f}")

In [ ]:
threshold = gross_array.mean()
above_avg_mask = gross_array >= threshold

print(f"Average worldwide gross: ${threshold:,.0f}")
print(f"Films above average:     {above_avg_mask.sum()} out of {len(gross_array)}")
print(f"Films below average:     {(~above_avg_mask).sum()}")
print()

print("Boolean mask (first 20):", above_avg_mask[:20])

In [ ]:
min_val = gross_array.min()
max_val = gross_array.max()
normalised = (gross_array - min_val) / (max_val - min_val)

df["gross_normalised"] = normalised.round(4)

print("Normalised gross (0–1 scale) — pandas ← NumPy ← pandas workflow:\n")
df[["title", "worldwide_gross", "gross_normalised"]].head(10)

In [ ]:
df["log_gross"] = np.log10(df["worldwide_gross"])

print("np.log10() applied to worldwide_gross:\n")
print(df[["title", "worldwide_gross", "log_gross"]].head(10).to_string(index=False))

---
## 11. Putting It All Together — Top Foreign Earners vs Top Domestic Earners

A short analysis pipeline that combines **filtering**, **sorting**, **column arithmetic**, and **`.isin()`**.

In [ ]:
valid = df[df["domestic_pct"].notna()].copy()

top20_by_foreign_rev = set(
    valid.sort_values("foreign_revenue", ascending=False).head(20)["title"]
)
top20_by_domestic_rev = set(
    valid.sort_values("domestic_revenue", ascending=False).head(20)["title"]
)

global_superstars = top20_by_foreign_rev & top20_by_domestic_rev
foreign_only_stars = top20_by_foreign_rev - top20_by_domestic_rev
domestic_only_stars = top20_by_domestic_rev - top20_by_foreign_rev

print(f"Top-20 foreign earners ∩ Top-20 domestic earners: {len(global_superstars)} films")
print(f"  Foreign-only stars:  {len(foreign_only_stars)}")
print(f"  Domestic-only stars: {len(domestic_only_stars)}")
print()

superstar_df = valid[valid["title"].isin(global_superstars)].sort_values(
    "worldwide_gross", ascending=False
)
print("=== Global Superstars (top earners in BOTH markets) ===\n")
print(superstar_df[["title", "worldwide_gross", "domestic_revenue", "foreign_revenue"]].to_string(index=False))

---
## 12. Reindexing and Dropping

Two handy DataFrame operations for reorganising data:

| Method | What it does |
|--------|-------------|
| `.reindex(new_labels)` | Rearrange rows to match new labels; inserts `NaN` for any missing |
| `.drop(labels)` | Remove rows or columns by label |

In [ ]:
sample = df.loc[0:4, ["title", "worldwide_gross", "domestic_pct"]].copy()
sample.index = ["a", "b", "c", "d", "e"]
print("Original sample:")
print(sample)
print()

print("Reindexed — 'f' is new, so it becomes NaN:")
print(sample.reindex(["e", "d", "c", "b", "a", "f"]))
print()

print("Drop the temporary helper columns we no longer need:")
cleaned = df.drop(columns=["domestic_share", "foreign_share", "title_length", "log_gross", "gross_normalised"])
print(f"Columns before: {len(df.columns)} → after: {len(cleaned.columns)}")
print(cleaned.columns.tolist())

---
## Summary — Session 7 Skills Practised

| Skill | Where we used it |
|-------|-----------------|
| **DataFrame creation** | `pd.DataFrame(raw_data)` — from a list of dicts (JSON) |
| **Inspection** | `.shape`, `.dtypes`, `.head()`, `.describe()` |
| **`.map(func)`** | Converting percentage strings to floats; classifying films into revenue tiers |
| **`.apply(func)`** | Calculating title lengths; computing column ranges |
| **Column arithmetic** | Domestic/foreign revenue, gross in billions |
| **Boolean filtering** | Billion-dollar club, foreign-dominant blockbusters, `~` negation |
| **Multiple conditions** | `&` and `|` with parentheses |
| **`.sort_values()`** | Sorting by domestic share, foreign share |
| **`.rank()`** | Gross rank vs domestic-share rank comparison |
| **`np.where()`** | Conditional `market_type` column |
| **`.value_counts()`** | Tier and market-type distributions |
| **`.unique()` / `.isin()`** | Distinct tiers; filtering by tier membership |
| **`.corr()`** | Correlation between domestic share and worldwide gross |
| **`.reindex()` / `.drop()`** | Reorganising and trimming the DataFrame |
| **NumPy integration** | `.to_numpy()`, vectorised stats, normalisation, `np.log10()` |